**Github repository:** https://github.com/NicholasBorch/ComSocSci-Assignments

**Group Members:** Nicholas Borch (s234841) and Robin Braagaard (s234856)

**Distribution of Contribution:** All group members contributed equally in the planning and execution of solutions and implementation of code for the assignment tasks.

------

# 02467 Computational social science - Assignment 1

------

### Python libraries used in assigment

In [95]:
### Importing libraries
from pathlib import Path
import pandas as pd
import re
import unicodedata
import time
from typing import Dict, Any, List, Optional, Tuple, Set
import numpy as np
import requests
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from unidecode import unidecode
from rapidfuzz import fuzz
from joblib import Parallel, delayed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from contextlib import contextmanager
import json

----

## Part 1: Ready Made vs Custom Made Data

> **Exercise: Ready made data vs Custom made data** In this exercise, I want to make sure you have understood they key points of my lecture and the reading. 

> 1. What are pros and cons of the custom-made data used in Centola's experiment (the first study presented in the lecture) and the ready-made data used in Nicolaides's study (the second study presented in the lecture)? You can support your arguments based on the content of the lecture and the information you read in Chapter 2.3 of the book __(answer in max 150 words)__.

#### Answer for P1Q1:

**Centola (Custom-made data):**

**Pros:** Strong causal clarity by having the network differences as the only causal variable. Experiment is designed explicitly around the research question. Participants self-opted into the platform following advertisements on health websites, not much suggests experiment awareness, suggesting a degree of non-reactivity.

**Cons:** Small scale (1,528 participants). Signing up to a forum is “low-cost”, genuine engagement is not captured; some may have signed up merely due to email annoyance. Study is unrepresentative, limited to internet users. Difficult to replicate as the platform is inaccessible to others.

**Nicolaides (Ready-made data):**

**Pros:** Massive scale (1.1M users, 350M+ km). Always-on continuous tracking over 5 years enables observation of emerging patterns. Based on real behaviour, not self-reports.

**Cons:** Non-representative, focusing on fitness-app users, probably an atypically health-motivated and competitive demographic, possibly exercising to beat friends rather than due to contagion. Algorithmic confounding likely, as the platform may artificially promote friend activity.



> 2. How do you think these differences can influence the interpretation of the results in each study? __(answer in max 150 words)__

#### Answer for P1Q2:

**Centola:** The small scale and artificial setting limit the generalisability of the experiment, results may not hold in larger, real-world networks. The low-cost behaviour that is being analyzed means we cannot conclude that clustered networks would similarly spread larger behavioural changes. However, the experimental control means the conclusions about network activity presented are highly credible within the study's scope.

**Nicolaides:** The massive scale and real-world setting strengthen generalisation. However, the non-representative sample of fitness-app users means results may not generalise too well to the rest of the population, which casts a few doubts on what the study finds the impact of contagion to be. Algorithmic confounding introduces uncertainty about whether observed contagion reflects genuine social influence or is being artificially pushed and constructed by the platform motivation. The reliance on behavioural data without further context means we cannot completely decide whether motivation or competitive pressure drives the observed behaviour.

------

## Pre Part 2 (Useful info and code from week 1):

Collecting the author names from the International Conference in Computational Social Science 2025.

Website URL: https://ic2s2-2025.org/program/

In [96]:
# Output folder for Assignment 1 datasets
A1_DATA_DIR = Path("A1_data")
A1_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Source CSV for Web-scraping the list of participants to the International Conference in Computational Social Science (found via Network tab in browser dev tools)
IC2S2_SCHEDULE_CSV_URL = "https://ic2s2-2025.org/files/ic2s2_2025_schedule_v5.csv"

In [97]:
# Setting up a function to canonicalize names, so that any spelling errors or differences in formatting 
# can be standardized for easier comparison and analysis and to avoid duplicates of the same person 
# being treated as different individuals.
def canon(name: str) -> str:
    """
    Canonical form of a person name for de-duplication:
    - lower-case
    - remove accents
    - transliterate remaining non-ASCII (ß->ss, ø->o, curly quotes->')
    - remove punctuation
    - collapse whitespace
    """
    if name is None:
        return ""
    name = str(name).strip().lower()
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = unidecode(name)
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name.strip()

In [98]:
### Loading IC2S2 schedule CSV
df_schedule = pd.read_csv(IC2S2_SCHEDULE_CSV_URL, storage_options={"User-Agent": "Mozilla/5.0"})
print("Rows:", len(df_schedule))
df_schedule.head()

Rows: 721


,date_start,date_end,type,title,abstract,author,location,session,session_order,chair,openreview_id,notes
0,2025-07-21 09:00:00,2025-07-21 12:30:00,single,Tutorial 1: LLM Power To The People,This tutorial aims to provide an up-to-date ov...,"Étienne Ollion, Émilien Schultz",Vingen 1+2,NaN,NaN,NaN,NaN,NaN
1,2025-07-21 09:00:00,2025-07-21 12:30:00,single,Tutorial 3: The Role of AI in Misinformation: ...,As AI-generated content becomes more prevalent...,"Miriam Schirmer, Julia Mendelsohn, Dustin Wrig...",Vingen 3+4,NaN,NaN,NaN,NaN,NaN
2,2025-07-21 09:00:00,2025-07-21 12:30:00,single,A Workflow for Open Reproducible Computational...,Reproducibility is essential for establishing ...,Caspar van Lissa,Vingen 7,NaN,NaN,NaN,NaN,NaN
3,2025-07-21 09:00:00,2025-07-21 12:30:00,single,Tutorial 7: Scalable Analysis of GPS Human Mob...,Large-scale human mobility datasets derived fr...,"Jorge Barreras, Thomas Li, Chen Zhong, Cate He...",Vingen 8,NaN,NaN,NaN,NaN,NaN
4,2025-07-21 09:00:00,2025-07-21 12:30:00,single,Tutorial 9: Reinforcement Learning and Evoluti...,Assuming that individuals are rational is ofte...,"Paolo Turrini, Elias Fernández Domingos",Vingen 6,NaN,NaN,NaN,NaN,NaN


In [99]:
### Extracting authors and chairs into one raw name series
authors = df_schedule["author"].dropna().astype(str).str.split(",").explode().str.strip()
authors = authors[authors.ne("")]

chairs = df_schedule["chair"].dropna().astype(str).str.split(",").explode().str.strip()
chairs = chairs[chairs.ne("")]

# Combining the two lists
names_raw = pd.concat([authors, chairs], ignore_index=True)

print("Raw author entries:", len(authors))
print("Raw chair entries :", len(chairs))
print("Raw combined entries:", len(names_raw))
print("Unique (raw, exact string):", names_raw.nunique())
names_raw.head(10)

Raw author entries: 2263
Raw chair entries : 285
Raw combined entries: 2548
Unique (raw, exact string): 1574


0          Étienne Ollion
1         Émilien Schultz
2         Miriam Schirmer
3        Julia Mendelsohn
4           Dustin Wright
5    Dietram A. Scheufele
6            Ágnes Horvát
7        Caspar van Lissa
8          Jorge Barreras
9               Thomas Li
dtype: object

In [100]:
# Creating a DataFrame that maps each raw name collected from the website 
# to its canonical version in order to standardize names and find 
# duplicates that may appear with slightly different spellings (e.g., accents, 
# punctuation differences, extra spaces). 
# Thereafter, counting how many unique canonical names exist (true unique researchers).
# and identify canonical names that correspond to multiple raw spellings.
names_df = pd.DataFrame({"name_raw": names_raw})
names_df["name_canon"] = names_df["name_raw"].map(canon)
print("Unique canonical names:", names_df["name_canon"].nunique())

# Canonical names that map to multiple raw spellings
variants = (names_df.groupby("name_canon")["name_raw"].unique().reset_index())
variants["n_variants"] = variants["name_raw"].apply(len)
variants_multi = variants[variants["n_variants"] > 1].sort_values("n_variants", ascending=False)

print("Canonical names with >1 raw variant:", len(variants_multi))
variants_multi

Unique canonical names: 1567
Canonical names with >1 raw variant: 7


,name_canon,name_raw,n_variants
321,daniel o brien,"[Daniel O’Brien, Daniel O'Brien]",2
415,elena d agnese,"[Elena D'Agnese, Elena D'agnese]",2
433,emilien schultz,"[Émilien Schultz, Emilien Schultz]",2
461,etienne ollion,"[Étienne Ollion, Etienne Ollion]",2
692,jia li,"[Jia LI, Jia Li]",2
825,kristina gligoric,"[Kristina Gligoric, Kristina Gligorić]",2
1191,rene algesheimer,"[Rene Algesheimer, René Algesheimer]",2


In [101]:
# Setting up the final list of unique IC2S2 researchers.
# Due to multiple raw spellings that map to the same canonical name,
# raw names are sorted alphabetically and only one representative
# raw name per canonical version is kept.
unique_researchers = (names_df.sort_values("name_raw").drop_duplicates("name_canon")["name_raw"].reset_index(drop=True))

out_unique = A1_DATA_DIR / "D0_unique_researchers.csv"
unique_researchers.to_csv(out_unique, index=False, header=["name"])

print("Unique researchers:", len(unique_researchers))
unique_researchers.head(20)

Unique researchers: 1567


0            Aakriti Kumar
1            Aaron Clauset
2          Aaron D Nichols
3             Aaron Reeves
4             Aaron Schein
5      Abdulaziz Alhumaidy
6       Abdullah Almaatouq
7             Abhisek Dash
8          Abraham Israeli
9           Achim Edelmann
10     Achyutarama R Ganti
11     Adam (Zhengzi) Zhou
12         Adam Stefkovics
13              Adel Daoud
14          Adeline Clarke
15      Adiba Mahbub Proma
16    Adolfo Fuentes-Jofre
17        Adrien Letellier
18        Adriënne Mendrik
19     Agnieszka Czaplicka
Name: name_raw, dtype: object

In [102]:
# In Week 2 (Part 2) text embeddings are used, hence a useful "text profile" is prepared for each person.

# For every row in the IC2S2 schedule (a talk/poster/etc.), a short document consisting of "title. abstract" is created.
# That document is then assigned to every author and chair listed on that row.

# This results in one text blob per researcher that summarizes what they contributed to at IC2S2 2025,
# and is used to find correct author ID in Part 2 of the assignment.
df_temp = df_schedule[["title", "abstract", "author", "chair"]].copy()
for col in ["title", "abstract", "author", "chair"]:
    df_temp[col] = df_temp[col].fillna("").astype(str)

rows = []
for _, r in df_temp.iterrows():
    paper_text = (r["title"].strip() + ". " + r["abstract"].strip()).strip()
    people = []
    people += [x.strip() for x in r["author"].split(",") if x.strip()]
    people += [x.strip() for x in r["chair"].split(",") if x.strip()]
    for person in people:
        rows.append({"name_raw": person, "name_canon": canon(person), "paper_text": paper_text})

df_people_text_long = pd.DataFrame(rows)

# Collecting all paper_text per person
df_people_text = (df_people_text_long.groupby("name_canon")["paper_text"].apply(lambda s: " ".join([t for t in s.tolist() if t])).reset_index().rename(columns={"paper_text": "conference_person_text"}))

out_person_text = A1_DATA_DIR / "conference_person_text.csv"
df_people_text.to_csv(out_person_text, index=False)

print("People with text:", len(df_people_text))
df_people_text.head()

People with text: 1567


,name_canon,conference_person_text
0,aakriti kumar,Evaluating Elements of Empathic Communication ...
1,aaron clauset,The Publication Preferences of Academics. This...
2,aaron d nichols,Truth Warrants Reduce Misleading Claims on Dig...
3,aaron reeves,Health By Wealth: Computational tools and popu...
4,aaron schein,Linear Representations of Political Perspectiv...


## Part 2: Find Researchers using the OpenAlex API

> **Exercise 3: Find potential Computational Social Scientists** In this exercise, we'll use the OpenAlex API to compile a list of researchers in the field of Computational Social Science, focusing on those who have attended the IC2S2 conference in 2025.

> 1. **Retreive data.** Consider the set of unique researcher names that you collected in Week 1, Exercise 3. Use the _authors_ endpoint of the [OpenAlex API](https://docs.openalex.org/api-entities/authors) to _search_ these researchers in the database based on their names. Loop through the list and, for each researcher in your list, find: 
>     - their _id_: The OpenAlex ID for this author.
>     - their _display\_name_: The name of the author as a single string.
>     - their _works\_api\_url_: A URL that will get you a list of all this author's works.
>     - their _h\_index_ : The h-index for this author.
>     - their _works\_count_: The number of  Works this this author has created.
>     - their _country\_code_: The country code of their last known institution

#### Answer for P2Q1:



OpenAlex API was used to collect author information.

OpenAlex API URL: https://docs.openalex.org/how-to-use-the-api/api-overview

In [ ]:
# OpenAlex API multi-key setup with matching email headers. After completing the Week 2 exercises, 
# OpenAlex seem to have reduced the free-tier credit limit. Hence, a multi key setup setup was opted 
# to in order to keep our original design with embedding similarity scores. The initial author-matching 
# pipeline was built up around the use of fuzzy name matching semantic embeddings, capturing context 
# appropriate authors from the OpenAlex database. The embeddings uses recent work titles per candidate to
# improve matching accuracy. This means each researcher lookup can trigger several API calls.

# To preserve this approach without hitting new budget limits mid-run, multiple free-tier API keys are utilized.
# Each key is paired with a personal email as required by OpenAlex's polite pool policy.

API_CREDENTIALS = [
    {"key": "OrnWTgBF4MDTq5Lj41EJoP", "email": "s234841@dtu.dk"},
    {"key": "5tTODa6zVIcgByCyqvxdrf", "email": "rbraagaard@gmail.com"},
    {"key": "zfogWMSQFJEZVu7vfoKtRJ", "email": "robinfifa0099@gmail.com"},
    {"key": "CvsqIRMWLKZ4XjnFzuhHKr", "email": "robinburner0099@gmail.com"},
    {"key": "y9jNIDioxpDJFcjmDdAjJm", "email": "s234856@dtu.dk"},
]

CURRENT_KEY_INDEX = 0

def get_current_api_key() -> str:
    return API_CREDENTIALS[CURRENT_KEY_INDEX]["key"]

def get_current_headers() -> Dict[str, str]:
    email = API_CREDENTIALS[CURRENT_KEY_INDEX]["email"]
    return {"User-Agent": f"IC2S2 Assignment (mailto:{email})"}

OPENALEX_AUTHORS_URL = "https://api.openalex.org/authors"
OPENALEX_WORKS_URL = "https://api.openalex.org/works"

# Embedding model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 418.21it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [104]:
### Loading D0 names and prepared conference_person_text
df_unique = pd.read_csv(A1_DATA_DIR / "D0_unique_researchers.csv")
df_person = pd.read_csv(A1_DATA_DIR / "conference_person_text.csv")

person_text_lookup = dict(zip(df_person["name_canon"].astype(str), df_person["conference_person_text"].astype(str)))
df_unique.head()

,name
0,Aakriti Kumar
1,Aaron Clauset
2,Aaron D Nichols
3,Aaron Reeves
4,Aaron Schein


In [105]:
# Name similarity scoring:

# OpenAlex often returns many candidate authors for a queried name,
# including both plausible matches and clearly unrelated individuals.
# We opted to first apply a fuzzy name similarity filter to ensure that only sufficiently
# similar names are considered. After this pre-filtering step, semantic embeddings of 
# research content is utilized to identify the most likely correct author.

def name_score(query: str, candidate: str) -> float:
    q = canon(query)
    c = canon(candidate)

    full = fuzz.token_set_ratio(q, c) / 100.0

    q_first = q.split()[0] if q.split() else ""
    c_first = c.split()[0] if c.split() else ""
    first = (fuzz.ratio(q_first, c_first) / 100.0) if (q_first and c_first) else 0.0

    return 0.7 * full + 0.3 * first

def rotate_api_key():
    global CURRENT_KEY_INDEX
    CURRENT_KEY_INDEX += 1
    if CURRENT_KEY_INDEX >= len(API_CREDENTIALS):
        raise RuntimeError("All API keys exhausted.")
    print(f"Switched to API key index {CURRENT_KEY_INDEX}")


def safe_get(
    url: str,
    params: Optional[dict] = None,
    retries: int = 6,
    base_wait: float = 1.2,
    timeout: Tuple[int, int] = (10, 45),  # (connect_timeout, read_timeout)
) -> Dict[str, Any]:
    """
    Robust GET with:
    - API key rotation on "Insufficient budget"
    - retry/backoff on 429 rate limit
    - retry/backoff on timeouts / transient connection issues
    """
    params = params or {}

    for attempt in range(1, retries + 1):
        params["api_key"] = get_current_api_key()
        headers = get_current_headers()

        try:
            r = requests.get(url, params=params, headers=headers, timeout=timeout)

            # Budget exhausted -> rotate key
            if r.status_code == 429 and "Insufficient budget" in r.text:
                rotate_api_key()
                continue

            # Normal 429 -> backoff and retry
            if r.status_code == 429:
                time.sleep(base_wait * (2 ** (attempt - 1)))
                continue

            # Other errors
            if r.status_code >= 400:
                raise RuntimeError(f"OpenAlex error {r.status_code}: {r.text[:300]}")

            return r.json()

        except (requests.exceptions.Timeout,
                requests.exceptions.ConnectionError,
                requests.exceptions.ChunkedEncodingError) as e:
            # Transient network / streaming issues -> retry with backoff
            time.sleep(base_wait * (2 ** (attempt - 1)))
            continue

    return {}

# Caches for speed and fewer API calls
author_search_cache: Dict[str, List[dict]] = {}
works_titles_cache: Dict[str, List[str]] = {}
conf_vec_cache: Dict[str, np.ndarray] = {}

In [ ]:
# This cell extracts exactly the fields required by the assignment:
#   - id
#   - display_name
#   - works_api_url
#   - h_index
#   - works_count
#   - country_code (from last_known_institution)

def search_authors(name: str, per_page: int = 100) -> List[dict]:
    if name in author_search_cache:
        return author_search_cache[name]

    data = safe_get(
        OPENALEX_AUTHORS_URL,
        params={
            "filter": f"display_name.search:{name}",
            "per-page": per_page,
        }
    )

    results = data.get("results", []) or []
    author_search_cache[name] = results
    return results

def author_id_short(author_obj: dict) -> str:
    aid = author_obj.get("id", "")
    return str(aid).rstrip("/").split("/")[-1] if aid else ""

def get_country_code(author_obj: dict) -> Optional[str]:
    lkis = author_obj.get("last_known_institutions")
    if isinstance(lkis, list) and lkis and isinstance(lkis[0], dict):
        return lkis[0].get("country_code")
    return None

def output_row_for_assignment(searched_name: str, author_obj: Optional[dict]) -> Dict[str, Any]:
    # Columns requested in Part 2
    if not isinstance(author_obj, dict):
        return {
            "searched_name": searched_name,
            "id": None,
            "display_name": None,
            "works_api_url": None,
            "h_index": None,
            "works_count": None,
            "country_code": None,
        }

    return {
        "searched_name": searched_name,
        "id": author_obj.get("id"),
        "display_name": author_obj.get("display_name"),
        "works_api_url": author_obj.get("works_api_url"),
        "h_index": (author_obj.get("summary_stats") or {}).get("h_index"),
        "works_count": author_obj.get("works_count"),
        "country_code": get_country_code(author_obj),
    }

In [107]:
# For the embedding model the last 5 works titles is fetched for author context.
def fetch_last_5_titles(author_obj: dict, max_titles: int = 5) -> List[str]:
    aid = author_id_short(author_obj)
    if not aid:
        return []

    if aid in works_titles_cache:
        return works_titles_cache[aid]

    data = safe_get(
        OPENALEX_WORKS_URL,
        params={
            "filter": f"authorships.author.id:{aid}",
            "sort": "publication_date:desc",
            "per-page": max_titles,
            "select": "title"
        }
    )

    results = data.get("results", []) or []

    titles = [
        w.get("title")
        for w in results
        if isinstance(w.get("title"), str) and w.get("title").strip()
    ]

    works_titles_cache[aid] = titles
    return titles

In [108]:
# Candidate embedding construction:
# For each OpenAlex author candidate, a semantic representation is built
# by concatenating (1) high-level topics, (2) more granular concepts, and
# (3) titles of the 15 most recent publications.

# If some components are missing, whatever information exists is embedded. 
# If all components are missing, semantic embedding is skipped for that candidate
# and instead we rely on the highest name similarity score.

def candidate_embedding_text(author_obj: dict) -> str:
    topics = " ".join([t.get("display_name", "") for t in (author_obj.get("topics", []) or []) if isinstance(t, dict)]).strip()

    concepts = " ".join([c.get("display_name", "") for c in (author_obj.get("x_concepts", []) or []) if isinstance(c, dict)]).strip()

    titles = fetch_last_5_titles(author_obj, max_titles=5)
    recent_works = " ".join([t for t in titles if isinstance(t, str) and t.strip()]).strip()

    return " ".join([topics, concepts, recent_works]).strip()

In [109]:
# Match one IC2S2 conference author

# For each conference name:
# 1) Qery OpenAlex to retrieve candidate authors.
# 2) Pre-filter candidates by fuzzy name similarity (name_score > 0.4) to remove clearly unrelated individuals. 
#    Threshold is chosen to still include names consisting of initials or abreviations (based on a small random sampling inspection in weekly exercises).
# 3) If the conference_person_text exists and at least one candidate has embeddable text (topics, concepts, recent works):
#    - Compute embeddings for both the conference text and each candidate.
#    - And select the candidate with the highest cosine similarity to the conference text.
#    Otherwise:
#       - Fall back to selecting the candidate with the highest name similarity score.

def get_conf_vec(name_canon: str, conf_text: str) -> Optional[np.ndarray]:
    if not conf_text or not str(conf_text).strip():
        return None
    if name_canon in conf_vec_cache:
        return conf_vec_cache[name_canon]
    v = model.encode(conf_text)
    conf_vec_cache[name_canon] = v
    return v

def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

def match_one_author(name_raw: str, conf_text: str) -> Optional[dict]:
    candidates = search_authors(name_raw)
    if not candidates:
        return None

    filtered: List[Tuple[float, dict]] = []

    for a in candidates:
        display = a.get("display_name", "")
        s = name_score(name_raw, display)
        if s > 0.4:
            filtered.append((s, a))

    if not filtered:
        return None

    filtered.sort(key=lambda x: x[0], reverse=True)

    # Keep maximum 15 best candidates for embedding
    filtered_top = filtered[:15]

    nc = canon(name_raw)
    conf_vec = get_conf_vec(nc, conf_text)

    if conf_vec is None:
        return filtered[0][1]

    emb_texts = []
    emb_authors = []

    for _, a in filtered_top:
        txt = candidate_embedding_text(a)
        if txt.strip():
            emb_texts.append(txt)
            emb_authors.append(a)

    if len(emb_authors) == 0:
        return filtered[0][1]

    cand_vecs = model.encode(emb_texts, batch_size=16)
    sims = [cosine(conf_vec, v) for v in cand_vecs]

    best_idx = int(np.argmax(sims))
    return emb_authors[best_idx]

> 2. **Data Storage** Store this information in a Pandas DataFrame and save it to file.

#### Answer for P2Q2:

In [110]:
# Creating the DataFrame with the required columns and save it as a D1 dataset in A1_data/.
# NOTE: Takes a long time to run.

D1_PATH = A1_DATA_DIR / "D1_openalex_authors.csv"

if D1_PATH.exists():
    df_existing = pd.read_csv(D1_PATH)
    processed_names = set(df_existing["searched_name"].astype(str))
    rows = df_existing.to_dict("records")
else:
    processed_names = set()
    rows = []

all_names = df_unique["name"].astype(str).tolist()

for name_raw in tqdm(all_names, desc="Part 2: OpenAlex author matching"):

    if name_raw in processed_names:
        continue

    try:
        conf_text = person_text_lookup.get(canon(name_raw), "")
        best = match_one_author(name_raw=name_raw, conf_text=conf_text)

        row = output_row_for_assignment(
            searched_name=name_raw,
            author_obj=best
        )

        rows.append(row)

        pd.DataFrame(rows).to_csv(D1_PATH, index=False)

    except RuntimeError as e:
        print(f"Stopped at name: {name_raw}")
        print(str(e))
        break

df_D1 = pd.DataFrame(rows)
df_D1.head()

Part 2: OpenAlex author matching: 100%|██████████| 1567/1567 [19:03<00:00,  1.37it/s] 


,searched_name,id,display_name,works_api_url,h_index,works_count,country_code
0,Aakriti Kumar,https://openalex.org/A5125610127,Aakriti Kumar,https://api.openalex.org/works?filter=author.i...,0.0,1.0,NaN
1,Aaron Clauset,https://openalex.org/A5014647140,Aaron Clauset,https://api.openalex.org/works?filter=author.i...,47.0,267.0,NaN
2,Aaron D Nichols,NaN,NaN,NaN,NaN,NaN,NaN
3,Aaron Reeves,https://openalex.org/A5039995516,Aaron Reeves,https://api.openalex.org/works?filter=author.i...,48.0,254.0,NaN
4,Aaron Schein,https://openalex.org/A5039072983,Aaron Schein,https://api.openalex.org/works?filter=author.i...,8.0,39.0,NaN


In [111]:
# Saving D1 dataset to file
D1_PATH = A1_DATA_DIR / "D1_openalex_authors.csv"
df_D1.to_csv(D1_PATH, index=False)
print("Saved:", D1_PATH)

Saved: A1_data\D1_openalex_authors.csv


> 3. **Reflection Questions** Answer the following questions __(max 150 words for each question)__: 
>
>    a) Which challenges did you encounter? How did you address them?

#### Answer for P2Q3a:

As briefly explained before, when matching researchers who attended the conference to OpenAlex authors, we faced the challenge that many researchers share the same name, we tried to solve this by constructing semantic embeddings, however this proved insufficient as many authors got matched to completely wrong names, which spurred us on to implement a two-step approach.

First, we applied a fuzzy name similarity filter (`name_score`) to remove clearly mismatched candidates while keeping plausible variations. Second, for the remaining candidates, we constructed semantic embeddings from each author’s research profile consisting of topics, concepts, and recent works, and compared them to embeddings of the attendee’s conference text. This allowed us to identify the candidate whose research content most closely matched the conference attendee.

If for some reason no textual information was available for embedding, we opted to just choose the candidate with the highest name similarity.


> 3.
>    b) Choose one problem you faced while collecting the data and describe your solution. Why did you choose this approach, and what impact might it have on your data? 

#### Answer for P2Q3b:

Even after implementing the two-step approach, visual inspections of the D1 dataset reveal that some names were matched to the incorrect author. This problem was attempted to be solved by adjusting the threshold of the fuzzy name score. A higher threshold reduces false positives but can remove the correct author if the name is abbreviated or spelled differently. A lower threshold increases recall but introduces more possible matches. Ultimately a relatively low threshold (0.4) was chosen, applied only to filter out clearly unrelated names, thereby relying on the embeddings to a greater extent.

Searching for vector embedding similarity should in theory be more precise than basic keyword lookup and standard name matching, hence this approach was prioritized. Mismatches are still present in the dataset, but this approach was deemed as being theoretically reliable and analytically robust.


--------

## Part 3: Collect Research Articles

> **Exercise 1: Collecting Research Articles from IC2S2 Authors**
>
>In this exercise, we'll leverage the OpenAlex API to gather information on research articles authored by participants of the IC2S2 2025 conference, referred to as *IC2S2 authors*.

> 1. **Retrieve Data:** Start with the dataset of *IC2S2 authors* you collected in Week 2, Exercise 3 (called dataset D1 in the figure above). Use the OpenAlex API [works endpoint](https://docs.openalex.org/api-entities/works) to fetch their research articles. For each article, retrieve the following details:
>    - _id_: The unique OpenAlex ID for the work.
>    - _publication_year_: The year the work was published.
>    - _cited_by_count_: The number of times the work has been cited by other works.
>    - _author_ids_: The OpenAlex IDs for the authors of the work.
>    - _title_: The title of the work.
>    - _abstract_inverted_index_: The abstract of the work, formatted as an inverted index.

> 2. **Data Storage:** Organize the retrieved information into two Pandas DataFrames and save them to two files in a suitable format:
>    - Dataset D2: The *IC2S2 papers* dataset should include: *id, publication\_year, cited\_by\_count, author\_ids*.
>    - Dataset D3: The *IC2S2 abstracts* dataset should include: *id, title, abstract\_inverted\_index*.

> **Filters:**
> To ensure the data we collect is relevant and manageable, apply the following filters:
>     
>    - Only include *IC2S2 authors* with a total work count between 5 and 5,000.    
>    - Retrieve only works that have received more than 10 citations.    
>    - Limit to works authored by fewer than 10 individuals.    
>    - Include only works relevant to Computational Social Science (focusing on: Sociology OR Psychology OR Economics OR Political Science) AND intersecting with a quantitative discipline (Mathematics OR Physics OR Computer Science), as defined by their [Concepts](https://docs.openalex.org/api-entities/works/work-object#concepts). *Note*: here we only consider Concepts at *level=0* (the most coarse definition of concepts).     
>
> **Efficiency Tips:**
> Writing efficient code in this exercise is **crucial**. To speed up your process:
> 
> - **Apply filters directly in your request:** When possible, use the [filter parameter](https://docs.openalex.org/api-entities/works/filter-works) of the *works* endpoint to apply the filters above directly in your API request, ensuring only relevant data is returned. Learn about combining multiple filters [here](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/filter-entity-lists).  
> - **Bulk requests:** Instead of sending one request for each author, you can use the [filter parameter](https://docs.openalex.org/api-entities/works/filter-works) to query works by multiple authors in a single request. *Note: My testing suggests that can only include up to 25 authors per request.*
> - **Use multiprocessing:** Implement multiprocessing to handle multiple requests simultaneously. I highly recommmend [Joblib’s Parallel](https://joblib.readthedocs.io/en/stable/) function for that, and [tqdm](https://tqdm.github.io/) can help monitor progress of your jobs. Remember to stay within [the rate limit](https://docs.openalex.org/how-to-use-the-api/rate-limits-and-authentication) of 100 requests per second.

#### Answer for P3Q1Q2:

In [112]:
OPENALEX_BASE = "https://api.openalex.org"
WORKS_ENDPOINT = f"{OPENALEX_BASE}/works"
CONCEPTS_ENDPOINT = f"{OPENALEX_BASE}/concepts"

PER_PAGE = 200                 # From assignment tip: use 200
AUTHOR_BATCH_SIZE = 25         # From assignment tip: safe bulk size
N_JOBS = 6                 
REQUEST_TIMEOUT = 30
MIN_SLEEP_BETWEEN_REQUESTS_SEC = 0.05

In [113]:
# Implementing an OpenAlex API client to reduce request failures and avoid rate limits,
# And setting up batch queries (chunking) and running of parallel joblib tasks.
def make_session() -> requests.Session:
    s = requests.Session()

    retry = Retry(
        total=6,
        backoff_factor=0.6,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )

    adapter = HTTPAdapter(max_retries=retry, pool_connections=50, pool_maxsize=50)
    s.mount("https://", adapter)

    # Dynamic headers per current key
    s.headers.update(get_current_headers())

    return s

def openalex_get(session: requests.Session, url: str, params: Dict[str, Any]) -> Dict[str, Any]:

    global CURRENT_KEY_INDEX

    params = dict(params)

    while True:

        params["api_key"] = get_current_api_key()
        session.headers.update(get_current_headers())

        r = session.get(url, params=params, timeout=REQUEST_TIMEOUT)

        # If key is exhausted then we rotate
        if r.status_code == 429 and "Insufficient budget" in r.text:
            rotate_api_key()
            session.headers.update(get_current_headers())
            continue

        # Normal rate limit
        if r.status_code == 429:
            time.sleep(1.5)
            continue

        if r.status_code >= 400:
            raise RuntimeError(f"OpenAlex error {r.status_code}: {r.text[:300]}")

        time.sleep(MIN_SLEEP_BETWEEN_REQUESTS_SEC)
        return r.json()

def author_id_short(openalex_author_id_url: str) -> str:
    if not isinstance(openalex_author_id_url, str):
        return ""
    return openalex_author_id_url.rstrip("/").split("/")[-1]

def work_id_short(openalex_work_id_url: str) -> str:
    if not isinstance(openalex_work_id_url, str):
        return ""
    return openalex_work_id_url.rstrip("/").split("/")[-1]

def chunk_list(xs: List[str], chunk_size: int) -> List[List[str]]:
    return [xs[i:i + chunk_size] for i in range(0, len(xs), chunk_size)]

@contextmanager
def tqdm_joblib(tqdm_object):
    import joblib
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_callback
        tqdm_object.close()

In [114]:
# Loading D1 (IC2S2 authors) dataset.
# Only including IC2S2 authors with total works_count between 5 and 5000.

D1_PATH = A1_DATA_DIR / "D1_openalex_authors.csv"
df_d1 = pd.read_csv(D1_PATH)

df_d1 = df_d1.copy()
df_d1["works_count"] = pd.to_numeric(df_d1["works_count"], errors="coerce")

df_d1_filt = df_d1[
    df_d1["id"].notna()
    & df_d1["works_count"].notna()
    & (df_d1["works_count"] >= 5)
    & (df_d1["works_count"] <= 5000)
].copy()

author_ids = [author_id_short(x) for x in df_d1_filt["id"].tolist()]
author_ids = [a for a in author_ids if a.startswith("A")]

author_batches = chunk_list(author_ids, AUTHOR_BATCH_SIZE)

In [115]:
# Looking up concept IDs at level=0
def get_level0_concept_id(session: requests.Session, concept_name: str) -> str:
    data = openalex_get(session, CONCEPTS_ENDPOINT, params={"search": concept_name, "per-page": 25})
    results = data.get("results", []) or []
    level0 = [c for c in results if c.get("level") == 0]
    target = concept_name.strip().lower()

    for c in level0:
        if str(c.get("display_name", "")).strip().lower() == target:
            return str(c["id"]).rstrip("/").split("/")[-1]
    return str(level0[0]["id"]).rstrip("/").split("/")[-1]

session = make_session()

css_names = ["Sociology", "Psychology", "Economics", "Political science"]
quant_names = ["Mathematics", "Physics", "Computer science"]

CSS_IDS = [get_level0_concept_id(session, n) for n in css_names]
QUANT_IDS = [get_level0_concept_id(session, n) for n in quant_names]

In [116]:
# Setting up works filter and row extraction for D2 and D3 datasets.
# D2 required columns: id, publication_year, cited_by_count, author_ids
# D3 required columns: id, title, abstract_inverted_index

def build_works_filter(author_ids_batch: List[str]) -> str:
    authors_or = "|".join(author_ids_batch)
    css_or = "|".join(CSS_IDS)
    quant_or = "|".join(QUANT_IDS)

    # Assignment filters applied when requesting:
    # - cited_by_count > 10
    # - authors_count < 10
    # - (CSS concepts OR) AND (Quant concepts OR)
    parts = [
        f"authorships.author.id:{authors_or}",
        "cited_by_count:>10",
        "authors_count:<10",
        f"concept.id:{css_or}",
        f"concept.id:{quant_or}",
    ]
    return ",".join(parts)

def extract_d2_d3_from_work(w: Dict[str, Any]) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    wid = work_id_short(w.get("id"))

    # author_ids
    a_ids: List[str] = []
    for a in (w.get("authorships") or []):
        aid_full = (a.get("author") or {}).get("id")
        if isinstance(aid_full, str) and aid_full:
            a_ids.append(author_id_short(aid_full))

    d2 = {
        "id": wid,
        "publication_year": w.get("publication_year"),
        "cited_by_count": w.get("cited_by_count"),
        "author_ids": "|".join([x for x in a_ids if x]),
    }

    d3 = {
        "id": wid,
        "title": w.get("title") if isinstance(w.get("title"), str) else "",
        "abstract_inverted_index": w.get("abstract_inverted_index"),
    }

    return d2, d3

In [117]:
# Fetching works for one author-batch using cursor paging
def fetch_works_for_author_batch(author_ids_batch: List[str]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    s = make_session()
    filt = build_works_filter(author_ids_batch)

    cursor = "*"
    d2_rows: List[Dict[str, Any]] = []
    d3_rows: List[Dict[str, Any]] = []

    while True:
        data = openalex_get(s, WORKS_ENDPOINT, params={"filter": filt, "per-page": PER_PAGE, "cursor": cursor})
        results = data.get("results", []) or []

        for w in results:
            d2, d3 = extract_d2_d3_from_work(w)
            if d2["id"]:
                d2_rows.append(d2)
                d3_rows.append(d3)

        next_cursor = (data.get("meta") or {}).get("next_cursor")
        if not next_cursor:
            break
        cursor = next_cursor

    return d2_rows, d3_rows

In [118]:
# Running parallel bulk collection across batches with joblib
pbar = tqdm(total=len(author_batches), desc="Part 3: /works author batches")

with tqdm_joblib(pbar):
    results = Parallel(n_jobs=N_JOBS, backend="loky")(delayed(fetch_works_for_author_batch)(batch) for batch in author_batches)

all_d2: List[Dict[str, Any]] = []
all_d3: List[Dict[str, Any]] = []

for d2_rows, d3_rows in results:
    all_d2.extend(d2_rows)
    all_d3.extend(d3_rows)

df_d2 = pd.DataFrame(all_d2).drop_duplicates(subset=["id"]).reset_index(drop=True)
df_d3 = pd.DataFrame(all_d3).drop_duplicates(subset=["id"]).reset_index(drop=True)

Part 3: /works author batches: 100%|██████████| 39/39 [00:29<00:00,  1.34it/s]


In [119]:
# Saving D2 as .csv and D3 jsonl
D2_PATH = A1_DATA_DIR / "D2_ic2s2_papers.csv"
D3_PATH = A1_DATA_DIR / "D3_ic2s2_abstracts.jsonl"

df_d2.to_csv(D2_PATH, index=False)

with open(D3_PATH, "w", encoding="utf-8") as f:
    for _, row in df_d3.iterrows():
        f.write(json.dumps(
            {
                "id": row["id"],
                "title": row["title"],
                "abstract_inverted_index": row["abstract_inverted_index"],
            },
            ensure_ascii=False
        ) + "\n")

> 3. **Data Overview and Reflection questions:** Answer the following questions __(max 150 words for each question)__: 
> 
> a) **Dataset summary.** How many works are listed in your Dataset D2 (*IC2S2 papers*) dataframe? How many unique researchers have co-authored these works?     

#### Answer for P3Q3a:

In [120]:
# Returning dataset summary D2 length and unique co-author count
D2_PATH = A1_DATA_DIR / "D2_ic2s2_papers.csv"
df_d2 = pd.read_csv(D2_PATH)

n_works = len(df_d2)

all_author_ids = (
    df_d2["author_ids"]
    .fillna("")
    .astype(str)
    .str.split("|")
    .explode()
    .str.strip()
)

unique_authors = set([a for a in all_author_ids.tolist() if a])
n_unique_authors = len(unique_authors)

print("Number of works in D2 (IC2S2 papers):", n_works)
print("Number of unique researchers co-authoring these works:", n_unique_authors)

Number of works in D2 (IC2S2 papers): 8991
Number of unique researchers co-authoring these works: 13666


### Output from cell above:

The number of works listed in Dataset D2 (IC2S2) is $8991$.

The number of unique researchers who have co-authered these works is $13666$

> b) **Efficiency in code.** Describe the strategies you implemented to make your code more efficient. How did your approach affect your code's execution time?

#### Answer for P3Q3b:

To make the code more efficient, a few different strategies were applied. First, bulk requests were used, meaning that multiple authors were queried at once instead of just sending one request per author. Second, multiprocessing with joblib was used, which allowed multiple batches of authors to be processed in parallel. This reduced the total runtime compared to running everything sequentially. Third, filters were applied directly in the API request, meaning only relevant works were retrieved instead of downloading unnecessary data and then only filtering afterwards. Finally, cursor paging was applied to safely and efficiently retrieve all results without missing or duplicating records. All this put together reduced the execution time by a fair amount and made sure that the data collection was able to be achieved in reasonable time.

> c) **Filtering Criteria and Dataset Relevance** Reflect on the rationale behind setting specific thresholds for the total number of works by an author, the citation count, the number of authors per work, and the relevance of works to specific fields. How do these filtering criteria contribute to the relevance of the dataset you compiled? Do you believe any aspects of Computational Social Science research might be underrepresented or overrepresented as a result of these choices?    

#### Answer for P3Q3c:

The filtering criteria were chosen to ensure the dataset is both relevant and easy to work with. The requirement of authors to have between 5-5000 total works removes researchers that are inactive while also removing those who might introduce noise. The citation threshold of 10+ citations ensures that research that has demonstrated academic impact is prioritized. Limiting papers to not having more than 10 authors avoids large-scale interdisciplinary collaborations, the rationale behind this being that such studies might extend beyond the specialized scope of Computational Social Science. Lastly, filtering by the chosen concepts ensures collected papers are relevant within the field. 

All these choices do however introduce some bias. Newer researchers and recently published works are probably not properly represented, as they are not cited sufficiently. Further, emerging and unexplored subfields within Computational Social Science may also be underrepresented, also because of the lack of citations and possibly large collaborations.

----